In [ ]:
import pandas as pd

df_raw = pd.read_excel("sample_address.xlsx")
df_raw = df_raw.rename(columns={"address": "raw_address"})
df_raw = df_raw.dropna(subset=["raw_address"])

In [ ]:
import re

alias_df = pd.read_csv("ph_address_alias_extended_v4.csv")

alias_map = dict(zip(alias_df["alias"], alias_df["standard"]))

def normalize_address(addr):
    addr = addr.lower()
    addr = re.sub(r"[^\w\s]", " ", addr)

    tokens = addr.split()
    tokens = [alias_map.get(tok, tok) for tok in tokens]

    return " ".join(tokens)

df_raw["normalized"] = df_raw["raw_address"].apply(normalize_address)

In [ ]:
dim = pd.read_csv("dim_location_20260415_v3.csv")

def retrieve_candidates(norm_addr):
    candidates = dim.copy()

    # coarse filtering
    for col in ["city_name", "province_name"]:
        matches = candidates[candidates[col].str.lower().apply(lambda x: x in norm_addr if isinstance(x, str) else False)]
        if len(matches) > 0:
            candidates = matches

    return candidates

df_raw["candidates"] = df_raw["normalized"].apply(retrieve_candidates)

In [ ]:
from rapidfuzz import fuzz

def score_candidate(norm_addr, row):
    score = 0

    if pd.notna(row["barangay_name"]):
        score += fuzz.partial_ratio(norm_addr, row["barangay_name"].lower()) * 0.4

    if pd.notna(row["city_name"]):
        score += fuzz.partial_ratio(norm_addr, row["city_name"].lower()) * 0.3

    if pd.notna(row["province_name"]):
        score += fuzz.partial_ratio(norm_addr, row["province_name"].lower()) * 0.3
    
    if pd.notna(row['region_name']):
        score += fuzz.partial_ratio(norm_addr, row["region_name"].lower()) * 0.3

    return score

def score_candidates(norm_addr, candidates):
    candidates = candidates.copy()
    candidates["score"] = candidates.apply(lambda row: score_candidate(norm_addr, row), axis=1)
    return candidates.sort_values(by="score", ascending=False)

df_raw["scored"] = df_raw.apply(lambda x: score_candidates(x["normalized"], x["candidates"]), axis=1)

In [ ]:
def select_best(scored_df):
    if len(scored_df) == 0:
        return None

    top = scored_df.iloc[0]

    return {
        "barangay": top["barangay_name"],
        "city": top["city_name"],
        "province": top["province_name"],
        "score": top["score"]
    }

df_raw["best_match"] = df_raw["scored"].apply(select_best)

In [ ]:
def qa_check(scored_df):
    if len(scored_df) == 0:
        return {"status": "NO_MATCH", "confidence": 0}

    top1 = scored_df.iloc[0]["score"]
    top2 = scored_df.iloc[1]["score"] if len(scored_df) > 1 else 0

    gap = top1 - top2

    if top1 < 70:
        return {"status": "LOW_CONFIDENCE", "confidence": top1}
    elif gap < 5:
        return {"status": "AMBIGUOUS", "confidence": top1}
    else:
        return {"status": "OK", "confidence": top1}

df_raw["qa"] = df_raw["scored"].apply(qa_check)

In [ ]:
def flatten(row):
    best = row["best_match"] or {}
    qa = row["qa"]

    return pd.Series({
        "raw_address": row["raw_address"],
        "normalized": row["normalized"],
        "barangay": best.get("barangay"),
        "city": best.get("city"),
        "province": best.get("province"),
        "score": best.get("score"),
        "qa_status": qa["status"],
        "confidence": qa["confidence"]
    })

df_output = df_raw.apply(flatten, axis=1)

df_output.to_excel("valid_ps_address.xlsx", index=False)